<a href="https://colab.research.google.com/github/zhengpohung/1d-tokenizer/blob/text_guide/text_guided_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install core packages (PyTorch and essential tools)
!pip install torch torchvision pandas pillow

# Install Transformer-related libraries
!pip install transformers accelerate sentencepiece protobuf

# Install configuration library
!pip install omegaconf

# Install Sionna and its dependencies
# Sionna requires a specific version of TensorFlow. The error log shows attempts to install compatible versions of dependencies.
# Let's ensure a compatible TensorFlow is installed before Sionna.
#!pip install tensorflow==2.19.1 # Use a recent stable version that is often compatible with modern libraries
!pip install piq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 8.6 MB/s eta 0:00:00


In [2]:
!pip install torchmetrics==0.11.4  # 確保版本相容性
!pip install lpips  # 感知損失庫

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.5 MB/s eta 0:00:00


In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Text_guided_TokCom'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"雲端硬碟根目錄設定為: {DRIVE_ROOT}")

Mounted at /content/drive
雲端硬碟根目錄設定為: /content/drive/MyDrive/Text_guided_TokCom


In [4]:
# 1. 強制解除安裝 Numpy (執行兩次以防有系統層級殘留)
!pip uninstall -y numpy
!pip uninstall -y numpy

# 2. 安裝與 Sionna 相容的特定版本組合
# 關鍵：同時限制 opencv 以免它自動把 numpy 升級回 2.0
!pip install "numpy==1.26.4" "opencv-python-headless<4.10" "scipy<1.13"

# 3. 重新安裝 TensorFlow 和 Sionna (確保它們抓到正確的 Numpy)
!pip install "tensorflow==2.19.1"
!pip install sionna

import torch
import numpy as np
import sionna
import itertools
import time
import os

# 修正 Sionna 相關的導入語句
from sionna.phy.fec.polar.encoding import Polar5GEncoder
from sionna.phy.fec.polar.decoding import Polar5GDecoder
# 修正：導入 CRCEncoder 和 CRCDecoder，而不是 CRC
from sionna.phy.fec.crc import CRCEncoder, CRCDecoder
from sionna.phy.utils import ebnodb2no
from sionna.phy.channel import AWGN
from sionna.phy.mapping import Mapper, Demapper

from transformers import AutoModelForCausalLM, AutoProcessor, CLIPTextModel, CLIPTokenizer, GenerationConfig
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

from omegaconf import OmegaConf

# 額外導入您在類別定義中使用的 LPIPS 和 CLIPScore
from piq import LPIPS
from torchmetrics.image.fid import FrechetInceptionDistance as FID
from torchmetrics.image import PeakSignalNoiseRatio as psnr
from torchmetrics.multimodal.clip_score import CLIPScore


Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 16.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 3.1 MB/s eta 0:00:00
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.19.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.4/520.4 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 24.9 

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
%cd /content/
!rm -rf /content/1d-tokenizer/
!git clone https://github.com/bytedance/1d-tokenizer.git
%cd 1d-tokenizer
!pip install -r requirements.txt
try:
    from modeling.tatitok import TATiTok
    from modeling.maskgen import MaskGen_VQ  # 注意這裡使用正確的 MaskGen_VQ
    from omegaconf import OmegaConf

    print("🎉 成功！所有模組已就緒。")
    print(f"TATiTok: {TATiTok}")
    print(f"MaskGen_VQ: {MaskGen_VQ}")

except ImportError as e:
    print(f"❌ 導入失敗: {e}")

In [ ]:
class WirelessImageTransmissionSystem:
  def __init__(self, device='cuda', drive_root='/content/drive/MyDrive/Text_guided_Tokcom'):
      self.device = device
      self.drive_root = drive_root
      print(f"初始化系統於 {device}...")

      # ---------------------------------------------------------
      # 1. 載入 AI 模型 (Molmo, CLIP, TA-TiTok, MaskGen)
      # ---------------------------------------------------------
      self._init_molmo()
      self._init_clip()
      self._init_tatitok_maskgen()
      self._init_metrics()

      # ---------------------------------------------------------
      # 2. 設置 5G PHY 層參數 (Sionna)
      # ---------------------------------------------------------
      # 論文設定：128 tokens, 8192 codebook -> 13 bits/token, 8 tokens/package, CRC11, N=256, 4-QAM
      self.tokens_per_image = 128
      self.bits_per_token = 13 # log2(8192)
      self.tokens_per_package = 8
      self.payload_bits = self.tokens_per_package * self.bits_per_token # 104
      self.crc_poly = "CRC11" # 5G NR Standard CRC11
      self.coder_n = 256 # 碼長

      # Sionna 組件
      self.crc_encoder = CRC_Encoder(self.crc_poly)
      self.crc_decoder = CRC_Decoder(self.crc_poly)

      # K (Polar 編碼輸入位元數) = Payload + CRC 長度
      self.k_polar = self.payload_bits + self.crc_encoder.crc_length # 104 + 11 = 115

      # 5G Polar Codec
      self.polar_encoder = Polar5GEncoder(k=self.k_polar, n=self.coder_n)
      self.polar_decoder = Polar5GDecoder(enc=self.polar_encoder, list_size=8)

      # 調變 (4-QAM)
      self.mapper = Mapper("qam", 4)
      self.demapper = Demapper("qam", 4, "app") # app = a posteriori probability (LLR)

      # 通道
      self.channel = AWGN()

  def _init_molmo(self):
      """初始化 Molmo-7B 用於圖像描述"""
      print("載入 Molmo-7B...")
      try:
          # 由於 Molmo 載入需要 transformers.GenerationConfig，我們從頂層導入 GenerationConfig
          self.molmo_processor = AutoProcessor.from_pretrained(
              "allenai/Molmo-7B-D-0924",
              trust_remote_code=True,
              torch_dtype=torch.float16, # 使用 float16 減少 VRAM 壓力
              device_map='auto'
          )
          self.molmo_model = AutoModelForCausalLM.from_pretrained(
              "allenai/Molmo-7B-D-0924",
              trust_remote_code=True,
              torch_dtype=torch.float16,
              device_map='auto'
          )
          self.use_molmo = True
          print("Molmo 載入成功。")
      except Exception as e:
          print(f"警告: Molmo 載入失敗 ({e})。將使用模擬描述。")
          self.use_molmo = False

  def _init_clip(self):
      """初始化 CLIP 用於文本編碼"""
      print("載入 CLIP...")
      self.clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
      # 為了節省 VRAM，我們將 CLIP 保持在 CPU 或與其他模型共享
      self.clip_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(self.device)

  def _init_tatitok_maskgen(self):
        """初始化 TA-TiTok 和 MaskGen"""
        print("載入 TA-TiTok 和 MaskGen...")

        # --- 配置檔路徑 (從專案目錄載入) ---
        # 根據您上傳的檔案結構，配置文件位於 configs/infer/TA-TiTok/ 和 configs/infer/MaskGen/
        CONFIG_PATH_TA = os.path.join("configs", "infer", "TA-TiTok", "tatitok_bl128_vq.yaml")
        CONFIG_PATH_MG = os.path.join("configs", "infer", "MaskGen", "maskgen_vq_xl.yaml")

        # --- 權重檔路徑 (從雲端硬碟載入) ---
        CKPT_PATH_TA = os.path.join(self.drive_root, "checkpoints", "tatitok_bl128_vq.bin")
        CKPT_PATH_MG = os.path.join(self.drive_root, "checkpoints", "maskgen_vq_xl.bin")

        try:
            # --- TA-TiTok 載入 ---
            self.tatitok_config = OmegaConf.load(CONFIG_PATH_TA) # <--- 載入本地配置
            self.tatitok = TATiTok(config=self.tatitok_config).to(self.device)
            self.tatitok.load_state_dict(torch.load(CKPT_PATH_TA, map_location=self.device)['state_dict']) # <--- 載入雲端權重
            self.tatitok.eval()
            print("TA-TiTok 載入成功。")

            # --- MaskGen 載入 ---
            self.maskgen_config = OmegaConf.load(CONFIG_PATH_MG) # <--- 載入本地配置
            self.maskgen = MaskGen_VQ(config=self.maskgen_config).to(self.device)
            self.maskgen.load_state_dict(torch.load(CKPT_PATH_MG, map_location=self.device)['state_dict']) # <--- 載入雲端權重
            self.maskgen.eval()
            print("MaskGen 載入成功。")

            self.vocab_size = self.tatitok_config.model.codebook_size
            self.mask_token_id = self.vocab_size

        except Exception as e:
            print(f"警告: TA-TiTok/MaskGen 載入失敗 ({e})。請檢查配置檔路徑和雲端硬碟中的權重檔案。")
            self.vocab_size = 8192
            self.mask_token_id = self.vocab_size
            pass

  def _init_metrics(self):
      """初始化評估指標"""
      self.lpips = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True).to(self.device)
      self.clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-large-patch14").to(self.device)
      print("Metric 模組已初始化。")

  def generate_caption(self, image):
      """使用 Molmo 生成描述 (Mocked 或 Real)"""
      if not self.use_molmo:
          return "A highly detailed image of a friendly golden retriever running on a sunny beach, captured in a photorealistic style."

      # 實際 Molmo 邏輯
      # 確保在 CPU 上載入的模型在推理時也使用 CPU 或正確的 device_map
      inputs = self.molmo_processor(
          images=image,
          text="Describe this image in detail.",
          return_tensors="pt"
      )

      output = self.molmo_model.generate(**inputs, max_new_tokens=100)
      generated_text = self.molmo_processor.tokenizer.decode(output[0, inputs.input_ids.size(1):], skip_special_tokens=True).strip()
      return generated_text

  def get_clip_embedding(self, caption):
      """將描述轉換為 77-token Embedding"""
      tokens = self.clip_tokenizer(
          caption,
          padding="max_length",
          max_length=77,
          truncation=True,
          return_tensors="pt"
      ).input_ids.to(self.device)

      with torch.no_grad():
          encoder_hidden_states = self.clip_encoder(tokens).last_hidden_state # [1, 77, 768]
      return encoder_hidden_states

  def int_to_bits(self, tokens):
      """Token IDs (int) 轉 bits"""
      masks = 1 << torch.arange(self.bits_per_token - 1, -1, -1).to(tokens.device)
      return ((tokens.unsqueeze(-1) & masks) > 0).int().flatten()

  def bits_to_int(self, bits):
      """bits 轉回 Token IDs"""
      bits = bits.view(-1, self.bits_per_token)
      masks = 1 << torch.arange(self.bits_per_token - 1, -1, -1).to(bits.device)
      return (bits * masks).sum(dim=1).int()

  @torch.no_grad()
  def transmission_channel(self, token_indices, snr_db):
      """模擬 5G PHY 通道傳輸"""

      num_packages = self.tokens_per_image // self.tokens_per_package
      token_packages = token_indices.view(num_packages, self.tokens_per_package)

      received_tokens_list = []
      package_error_flags = []

      for i in range(num_packages):
          package = token_packages[i]

          # --- 發射端 ---
          u = self.int_to_bits(package).float().unsqueeze(0) # [1, 104]
          c = self.crc_encoder(u) # [1, 115]
          x = self.polar_encoder(c) # [1, 256]
          x_sym = self.mapper(x)

          # --- 通道 (AWGN) ---
          ebn0_bit = snr_db + 10 * torch.log10(torch.tensor(self.k_polar / self.coder_n * 2))
          no = ebnodb2no(ebn0_bit, 2, self.k_polar/self.coder_n)

          y = self.channel([x_sym, no])

          # --- 接收端 ---
          llr = self.demapper([y, no])
          c_hat = self.polar_decoder(llr)
          u_hat, crc_valid = self.crc_decoder(c_hat)

          rec_tokens = self.bits_to_int(u_hat.squeeze(0))

          is_error = not crc_valid.item()

          if is_error:
              # CRC 檢查失敗，整個封包標記為 [MASK]
              masked_package = torch.full_like(rec_tokens, self.mask_token_id)
              received_tokens_list.append(masked_package)
              package_error_flags.extend([1] * self.tokens_per_package)
          else:
              received_tokens_list.append(rec_tokens)
              package_error_flags.extend([0] * self.tokens_per_package)

      # 重組
      final_tokens = torch.cat(received_tokens_list)
      error_mask = torch.tensor(package_error_flags).to(self.device)

      return final_tokens, error_mask

  @torch.no_grad()
  def run_simulation_step(self, original_image, image_caption, snr_db):
      """運行單一步驟模擬，並返回原始圖像、重建圖像和評估指標"""

      # 0. 數據轉換
      img_tensor_norm = transforms.ToTensor()(original_image).float().unsqueeze(0).to(self.device)

      # 1. 文本 Embedding
      text_embeddings = self.get_clip_embedding(image_caption)

      # 2. 圖像 Tokenization (TA-TiTok)
      if hasattr(self, 'tatitok'):
          # 實際 TA-TiTok 邏輯
          token_indices = self.tatitok.encode(img_tensor_norm)['indices'][0]
      else:
          # Mock
          token_indices = torch.randint(0, self.vocab_size, (self.tokens_per_image,)).to(self.device)

      # 3. 無線傳輸
      received_tokens_masked, error_mask = self.transmission_channel(token_indices, snr_db)

      # 4. Token 重建 (MaskGen)
      if hasattr(self, 'maskgen'):
          # 實際 MaskGen 邏輯
          # MaskGen.sample 需要知道要預測多少 token，通常是 12 次迭代
          reconstructed_tokens = self.maskgen.sample(received_tokens_masked, text_embeddings, num_steps=12)
      else:
          # Mock
          reconstructed_tokens = received_tokens_masked.clone()

      # 5. De-tokenization (TA-TiTok)
      if hasattr(self, 'tatitok'):
          # 實際 TA-TiTok decode 邏輯
          recon_image_tensor = self.tatitok.decode(reconstructed_tokens.unsqueeze(0), text_embeddings).clamp(0, 1)
      else:
          # Mock
          recon_image_tensor = transforms.GaussianBlur(5)(img_tensor_norm)


      # 評估指標
      if hasattr(self, 'lpips'):
          # 實際計算
          psnr = psnr_metric(img_tensor_norm, recon_image_tensor, data_range=1.0).item()
          lpips = self.lpips(img_tensor_norm, recon_image_tensor).item()
          clip_score = self.clip_metric(recon_image_tensor, image_caption).item() / 100.0 # 轉換為 0-1 範圍
      else:
          # Mocking Metric Results for demonstration
          per = error_mask.float().mean().item()
          psnr = 25 - snr_db * 0.5
          lpips = 0.1 + snr_db * 0.02
          clip_score = 0.8 - snr_db * 0.01

      return {
          'per': error_mask.float().mean().item(),
          'psnr': psnr,
          'lpips': lpips,
          'clip': clip_score,
          'original_img': original_image,
          'recon_img_tensor': recon_image_tensor
      }

In [ ]:
import json
import os

# 1. 初始化系統 (Colab 有 GPU，正常載入 Molmo)
# 確保您已經執行了前面的安裝和 class 定義
sim_system = WirelessImageTransmissionSystem(device='cuda', drive_root='/content/drive/MyDrive/Text_guided_TokCom')

# 2. 準備數據集 (這裡假設您有 ImageNetLoader)
dataset = ImageNetLoader(root_dir=os.path.join(DRIVE_ROOT, 'val.X'), size=5)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

caption_data = {}

print("開始生成圖片描述...")
for idx, (image_tensor, _) in enumerate(dataloader):
    # 注意：ImageNetLoader如果是回傳 Tensor，可能需要轉回 PIL Image 給 Molmo
    # 這裡假設您的 loader 或處理邏輯能拿到 PIL Image
    # image_pil = transforms.ToPILImage()(image_tensor[0])

    # 或是直接使用您的 image_pil (如果 loader 有修改過)
    # 假設這裡 image_pil 是當前的圖片物件

    # 生成描述
    caption = sim_system.generate_caption(image_pil)

    # 儲存結果 (使用索引或檔名作為 Key)
    image_key = f"image_{idx}"
    caption_data[image_key] = caption
    print(f"Image {idx} Caption: {caption[:50]}...")

    if idx >= 5: break # 測試用

# 3. 儲存成 JSON 檔案
output_file = 'image_captions.json'
with open(output_file, 'w') as f:
    json.dump(caption_data, f, indent=4)

print(f"描述已儲存至 {output_file}，請下載此檔案到本地端。")
from google.colab import files
files.download(output_file)

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
sim_system = WirelessImageTransmissionSystem(device=DEVICE, drive_root=DRIVE_ROOT)
snr_range_db = np.arange(-5.0, 5.1, 1.0)
print(f"從 {DRIVE_ROOT} 載入 ImageNet 模擬數據集...")
dataset = ImageNetLoader(root_dir=os.path.join(DRIVE_ROOT, 'val.X'), size=5)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

results = {}

for snr_db in snr_range_db:
  print(f"\n--- 模擬 SNR: {snr_db:.1f} dB ---")

  snr_results = {
      'psnr_list': [],
      'lpips_list': [],
      'clip_list': [],
      'per_list': []
  }

  for idx, (image, caption) in enumerate(dataloader):
      if idx >= 5: break # 僅處理前 5 張圖片以加快模擬速度

      image_pil = image[0]
      caption_text = caption[0]

      # 1. 產生描述
      final_caption = sim_system.generate_caption(image_pil)

      # 2. 執行單步模擬
      step_result = sim_system.run_simulation_step(image_pil, final_caption, snr_db)

      print(f"  Img {idx}: PER={step_result['per']:.2%}, PSNR={step_result['psnr']:.2f}, CLIP={step_result['clip']:.4f}")

      # 收集結果
      snr_results['psnr_list'].append(step_result['psnr'])
      snr_results['lpips_list'].append(step_result['lpips'])
      snr_results['clip_list'].append(step_result['clip'])
      snr_results['per_list'].append(step_result['per'])

  # 計算平均值
  results[snr_db] = {
      'avg_psnr': np.mean(snr_results['psnr_list']),
      'avg_lpips': np.mean(snr_results['lpips_list']),
      'avg_clip': np.mean(snr_results['clip_list']),
      'avg_per': np.mean(snr_results['per_list']),
  }


  print("\n--- 模擬結果彙總 (Average over samples) ---")
  print("SNR(dB) | Avg_PER | Avg_PSNR | Avg_LPIPS | Avg_CLIP")
  print("-----------------------------------------------------")
for snr, res in results.items():
  print(f"{snr:7.1f} | {res['avg_per']:.4f} | {res['avg_psnr']:.4f} | {res['avg_lpips']:.4f} | {res['avg_clip']:.4f}")